<a href="https://colab.research.google.com/github/CatherineMatangu/MyRepo_2026_Analytics2/blob/main/DimentionalityReductionAndOLAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Seed for reproducibility
np.random.seed(42)

# --- 1. Simulate Healthcare Data ---

# Define parameters for simulation
num_clinics = 5
num_days = 6 * 30 # 6 months of data

# Clinic names
clinic_names = [f'Clinic_{i+1}' for i in range(num_clinics)]

# Dates over 6 months
dates = pd.date_range(start='2025-01-01', periods=num_days, freq='D')

# Common ailments (seasonal variation can be added later for OLAP)
ailments = ['Flu', 'Malaria', 'Typhoid', 'Pneumonia', 'Diarrhea', 'Common Cold', 'Asthma', 'Diabetes', 'Hypertension']

# Medication types
medications = ['Paracetamol', 'Amoxicillin', 'Antimalarial', 'Insulin', 'Ventolin', 'Antibiotic_X', 'Painkiller_Y']

# Initialize lists to store data
data = []

# Corrected probabilities for ailments to sum to 1
ailment_probabilities = [0.15, 0.1, 0.08, 0.07, 0.12, 0.15, 0.1, 0.1, 0.13] # Sums to 1.0

for clinic in clinic_names:
    for date in dates:
        # Clinic Attendance (simulated to vary)
        attendance = np.random.randint(50, 200) # 50 to 200 patients per day

        # Ailments (multiple ailments per day, with varying severity/count)
        daily_ailments = np.random.choice(ailments, size=np.random.randint(1, 5), p=ailment_probabilities)
        ailment_counts = {ail: np.random.randint(5, attendance // 5) for ail in daily_ailments} # Count for each ailment

        # Medication Supply (simulated consumption/dispensing)
        daily_medication_supply = {med: np.random.randint(10, 100) for med in np.random.choice(medications, size=np.random.randint(2, 6), replace=False)}

        row = {
            'Clinic': clinic,
            'Date': date,
            'Attendance': attendance,
        }
        row.update({f'Ailment_{a}': ailment_counts.get(a, 0) for a in ailments})
        row.update({f'Medication_{m}_Supply': daily_medication_supply.get(m, 0) for m in medications})
        data.append(row)

df = pd.DataFrame(data)

# Save the simulated raw data (optional, for inspection)
df.to_csv('simulated_healthcare_data_raw.csv', index=False)
print("Simulated raw healthcare data saved to simulated_healthcare_data_raw.csv")

# --- 2. Dimensionality Reduction (PCA) ---

# Select features for PCA: Ailment counts and Medication Supply
features = [col for col in df.columns if 'Ailment_' in col or 'Medication_' in col]
X = df[features]

# Standardize the features before applying PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
# We'll choose a number of components that explains a significant portion of the variance, e.g., 95%
pca = PCA(n_components=0.95) # Retain 95% of variance
X_reduced = pca.fit_transform(X_scaled)

# Create a DataFrame for the reduced data
df_reduced = pd.DataFrame(X_reduced, columns=[f'PC_{i+1}' for i in range(X_reduced.shape[1])])

# Add back 'Clinic' and 'Date' for context in OLAP
df_reduced['Clinic'] = df['Clinic']
df_reduced['Date'] = df['Date']

# Save the reduced dataset
df_reduced.to_csv('simulated_healthcare_data_reduced.csv', index=False)
print(f"Dimensionality reduced data saved to simulated_healthcare_data_reduced.csv with {pca.n_components_} components.")

# Report explained variance
explained_variance_ratio = pca.explained_variance_ratio_
print("Explained variance ratio per component:", explained_variance_ratio)
print("Total explained variance by selected components:", sum(explained_variance_ratio))

# Display head of the reduced data
print("\nHead of the reduced dataset:")
print(df_reduced.head())

Simulated raw healthcare data saved to simulated_healthcare_data_raw.csv
Dimensionality reduced data saved to simulated_healthcare_data_reduced.csv with 15 components.
Explained variance ratio per component: [0.07692054 0.0741406  0.07217494 0.07082845 0.07025825 0.06852572
 0.06735743 0.06362163 0.06089675 0.05941534 0.05767154 0.05645397
 0.05483809 0.05196714 0.04757678]
Total explained variance by selected components: 0.9526471505968135

Head of the reduced dataset:
       PC_1      PC_2      PC_3      PC_4      PC_5      PC_6      PC_7  \
0  3.760243 -0.497850 -0.760831 -1.101663  1.677631 -0.231140  0.485334   
1 -1.396143 -0.095452 -0.091565 -0.691828 -0.514336 -0.568702 -0.957565   
2 -1.136971 -0.184056 -1.093073 -1.961226  0.180957 -0.204002 -1.110008   
3 -2.229924 -0.277026 -0.984984  0.484225  1.056099 -0.178834 -0.274962   
4  0.934162 -0.536773  1.420702 -0.524928  1.446159 -0.302821 -0.370946   

       PC_8      PC_9     PC_10     PC_11     PC_12     PC_13     PC_14  \